In [2]:
from ultralytics import YOLO
import os
import glob
import cv2
import json
import pandas as pd
import argparse
print(os.getcwd())

/DATA/jhlee_temp/pjw/ultralytics


In [ ]:
# Load a pretrained YOLO11n model
model_path = "./runs/detect/train6/weights/best.pt"
model = YOLO(model_path)
source = "./CytologIA_Dataset/images/test/"
# Run inference on 'bus.jpg' with arguments
model.predict(source=source, save_txt=True, imgsz=320, batch=4, conf=0.15, device=3, project="./run", name="predict")

In [ ]:
class_names = {
    0: 'PNN',
    1: 'MM',
    2: 'LyB',
    3: 'LGL',
    4: 'Thromb',
    5: 'LLC',
    6: 'LAM3',
    7: 'EO',
    8: 'LY',
    9: 'BA',
    10: 'MoB',
    11: 'LM',
    12: 'MO',
    13: 'LH_lyAct',
    14: 'Lysee',
    15: 'Er',
    16: 'LF',
    17: 'LZMG',
    18: 'SS',
    19: 'MBL',
    20: 'PM',
    21: 'B',
    22: 'M'
}

In [ ]:

def yolo_to_bbox(yolo_data, image_width, image_height):

    bbox_data = []
    for row in yolo_data:
        class_id, cx, cy, w, h = map(float, row.strip().split())
        x1 = (cx - w / 2) * image_width
        y1 = (cy - h / 2) * image_height
        x2 = (cx + w / 2) * image_width
        y2 = (cy + h / 2) * image_height
        bbox_data.append({
            "x1": int(x1),
            "y1": int(y1),
            "x2": int(x2),
            "y2": int(y2),
            "class": class_names[int(class_id)]
        })
    return bbox_data

In [ ]:
def test_csv_create(output_file, img_path, pred_path, types):
    
    test_df = pd.read_csv(output_file)
    pred_data = []

    prediction_files = glob.glob(os.path.join(pred_path, "*.txt"))

    for i, pred_file in enumerate(prediction_files) :
        file_name = os.path.basename(pred_file).split('.')[0]
        img_file = os.path.join(img_path, f"{file_name}.jpg")

        if not os.path.exists(img_file):
            print(f"Warning: Image file {img_file} not found.")
            continue

        with open(pred_file, 'r') as f:
            yolo_data = f.readlines()

        img = cv2.imread(img_file)
        if img is None:
            print(f"Warning: Failed to load image {img_file}.")
            continue

        image_height, image_width = img.shape[:2]
        bbox_data = yolo_to_bbox(yolo_data, image_width, image_height)
        print(f"{i}:{file_name}:{bbox_data}")
        for bbox in bbox_data:
            bbox["NAME"] = file_name + ".jpg"
            pred_data.append(bbox)
    pred_df = pd.DataFrame(pred_data)
        
    pred_df['row_id'] = pred_df.groupby('NAME').cumcount()
    test_df['row_id'] = test_df.groupby('NAME').cumcount()

    results = pd.merge(test_df, pred_df, on=["NAME", "row_id"], how="left")
    results = results.drop(columns=['row_id'])
    results = results.fillna(0)

    # Save combined results to CSV
    results.to_csv("./cytologia-data-1732098640162_test.csv",index=False)

In [ ]:
img_path="./CytologIA_Dataset/images/test/"
pred_path="./ultralytics/run/predict/labels"
output_file='./cytologia-data-1732098640162.csv'
    
test_csv_create(output_file, img_path,pred_path)